# Proyecto GestoVoz 
Prof. Adan Hirales
Equipo: 
- Christofer Castaneda
- Ivan Mijares
- Alexander Orduna

# Librerias y Ruta 

In [ ]:
# solo en caso de EMERGENCIAs descomentar
# import sys
# !{sys.executable} -m pip install \
#     "numpy==1.26.4" \
#     "opencv-python==4.8.1.78" \
#     "mediapipe==0.10.9" \
#     "seaborn" \
#     "scikit-learn==1.5.1" \
#     "coremltools" \
#     --force-reinstall

import cv2
import mediapipe as mp
import numpy as np
import joblib
import json
import time
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import recall_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Gestos
GESTURE_MAP = {
    "dislike": 0,
    "fist"   : 1,
    "like"   : 2,
    "ok"     : 3,
    "one"    : 4,
    "palm"   : 5,
    "peace"  : 6,
    "rock"   : 7,
}

LABEL_NAMES = list(GESTURE_MAP.keys())
N_CLASSES   = len(GESTURE_MAP)
N_LANDMARKS = 21
N_COORDS    = 2
N_FEATURES  = N_LANDMARKS * N_COORDS + 5

MVP_F1_MACRO     = 0.88
MVP_ACCURACY     = 0.90
MVP_ALERT_RECALL = 0.90
ALERT_CLASS      = 5  # palm

print(f"Clases  : {N_CLASSES}")
print(f"Features: {N_FEATURES}")
for name, idx in GESTURE_MAP.items():
    alert = " <- ALERTA" if idx == ALERT_CLASS else ""
    print(f"  [{idx}] {name}{alert}")

In [ ]:
HAGRID_ROOT = Path(r"D:\hagrid_normalized")
CACHE_DIR   = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

CACHE_X    = CACHE_DIR / "X_landmarks.npy"
CACHE_Y    = CACHE_DIR / "y_labels.npy"
MODEL_PATH = "gestovoz_rf.pkl"
META_PATH  = "gestovoz_metadata.json"

print("Debug de direccion, clases y features")
print(f"Dataset    : {HAGRID_ROOT}")
print(f"Clases     : {N_CLASSES}")
print(f"Features   : {N_FEATURES}")
print(f"\n   Mapeo de gestos:")
for name, idx in GESTURE_MAP.items():
    print(f"     [{idx}] {name}")

# Extraccion de features

Para cada imagen del data set de HaGRID:
  1. Detectar la mano con MediaPipe Hands
  2. Extraer 21 landmarks (x, y, z)
  3. Normalizar relativo a la muneca (landmark 0)
  4. Guardar vector de 63 floats + etiqueta

Si ya existe el cache, se salta la extraccion automaticamente.

In [ ]:
mp_hands   = mp.solutions.hands
EXTENSIONS = {".jpg", ".jpeg", ".png"}

def angle_between(v1, v2):
    dot  = np.dot(v1, v2)
    norm = np.linalg.norm(v1) * np.linalg.norm(v2)
    if norm < 1e-6:
        return 0.0
    return np.arccos(np.clip(dot / norm, -1.0, 1.0))

def normalize_landmarks_xy(hand_landmarks) -> list[float]:
    raw_x = [lm.x for lm in hand_landmarks.landmark]
    raw_y = [lm.y for lm in hand_landmarks.landmark]

    # Centrar en muneca
    wx, wy = raw_x[0], raw_y[0]
    cx = [x - wx for x in raw_x]
    cy = [y - wy for y in raw_y]

    # Normalizar escala
    scale = np.sqrt(cx[9]**2 + cy[9]**2)
    if scale < 1e-6:
        scale = 1.0

    # 42 coordenadas normalizadas
    coords = []
    for x, y in zip(cx, cy):
        coords.append(x / scale)
        coords.append(y / scale)

    # 5 angulos entre puntas de dedos consecutivos
    fingertips = [4, 8, 12, 16, 20]
    angles = []
    for i in range(len(fingertips) - 1):
        a  = fingertips[i]
        b  = fingertips[i + 1]
        v1 = np.array([cx[a] / scale, cy[a] / scale])
        v2 = np.array([cx[b] / scale, cy[b] / scale])
        angles.append(angle_between(v1, v2))

    return coords + angles  # 47 floats


#### Extraer el dataset
Recorre hagrid_root/gesto/*.jpg y extrae los landmarks.
- hagrid_root- carpeta raiz (Hagrid) con subcarpetas por gesto (8 carpetas)
- gesture_map- dict nombre_gesto -> int label
- max_per_class- limite de imagenes por clase (None = todas)
- min_detection_confidence- La minima 

In [ ]:
def extract_dataset(hagrid_root, gesture_map, max_per_class=None):
    X, y = [], []
    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:
        for gesture_name, label in gesture_map.items():
            gesture_dir = hagrid_root / gesture_name
            images = sorted([
                p for p in gesture_dir.iterdir()
                if p.suffix.lower() in EXTENSIONS
            ])
            if max_per_class:
                images = images[:max_per_class]

            detected, skipped = 0, 0
            for img_path in tqdm(images, desc=f"[{label}] {gesture_name:<8s}", unit="img"):
                img_bgr = cv2.imread(str(img_path))
                if img_bgr is None:
                    skipped += 1
                    continue
                result = hands.process(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
                if result.multi_hand_landmarks:
                    feats = normalize_landmarks_xy(result.multi_hand_landmarks[0])
                    X.append(feats)
                    y.append(label)
                    detected += 1
                else:
                    skipped += 1
            print(f"  {detected:,} detectadas | {skipped:,} sin mano\n")

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_image(args):
    img_path, label, = args
    mp_h = mp.solutions.hands
    with mp_h.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            return None
        result = hands.process(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        if result.multi_hand_landmarks:
            feats = normalize_landmarks_xy(result.multi_hand_landmarks[0])
            return (feats, label)
        return None

def extract_dataset_parallel(hagrid_root, gesture_map, max_per_class=None, workers=8):
    tasks = []
    for gesture_name, label in gesture_map.items():
        gesture_dir = hagrid_root / gesture_name
        images = sorted([
            p for p in gesture_dir.iterdir()
            if p.suffix.lower() in EXTENSIONS
        ])
        if max_per_class:
            images = images[:max_per_class]
        for img_path in images:
            tasks.append((img_path, label))

    X, y = [], []
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(process_image, t): t for t in tasks}
        for future in tqdm(as_completed(futures), total=len(tasks), desc="Extrayendo"):
            result = future.result()
            if result is not None:
                X.append(result[0])
                y.append(result[1])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

In [ ]:
if CACHE_X.exists() and CACHE_Y.exists():
    print("Cache encontrado")
    X_data = np.load(CACHE_X)
    y_data = np.load(CACHE_Y)
    print(f"{X_data.shape[0]:,} muestras | {X_data.shape[1]} features")
else:
    print("Extrayendo landmarks...\n")
    X_data, y_data = extract_dataset_parallel(
        HAGRID_ROOT,
        GESTURE_MAP,
        max_per_class=None,
        workers=8        # prueba con 4, si va bien sube a 6 u 8
    )
    np.save(CACHE_X, X_data)
    np.save(CACHE_Y, y_data)
    print(f"Cache guardado en '{CACHE_DIR}'")

# Balanceo
MAX_PER_CLASS = 14_000

def balance_dataset(X, y, max_per_class, seed=42):
    rng     = np.random.default_rng(seed)
    indices = []
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        if len(idx) > max_per_class:
            idx = rng.choice(idx, size=max_per_class, replace=False)
        indices.append(idx)
    indices = np.concatenate(indices)
    rng.shuffle(indices)
    return X[indices], y[indices]

X_data, y_data = balance_dataset(X_data, y_data, MAX_PER_CLASS)

print(f"\nDataset balanceado: {X_data.shape[0]:,} muestras totales")
for name, idx in GESTURE_MAP.items():
    count = np.sum(y_data == idx)
    print(f"  [{idx}] {name:<8s}  {count:,}")

In [ ]:
# if CACHE_X.exists() and CACHE_Y.exists():
#     print("Cache encontrado")
#     X_data = np.load(CACHE_X)
#     y_data = np.load(CACHE_Y)
#     print(f"{X_data.shape[0]:,} muestras | {X_data.shape[1]} features")
# else:
#     print("Extrayendo landmarks...\n")
#     X_data, y_data = extract_dataset(HAGRID_ROOT, GESTURE_MAP, max_per_class=None)
#     np.save(CACHE_X, X_data)
#     np.save(CACHE_Y, y_data)
#     print(f"Cache guardado en '{CACHE_DIR}'")

# # Balanceo
# MAX_PER_CLASS = 14_000

# def balance_dataset(X, y, max_per_class, seed=42):
#     rng     = np.random.default_rng(seed)
#     indices = []
#     for label in np.unique(y):
#         idx = np.where(y == label)[0]
#         if len(idx) > max_per_class:
#             idx = rng.choice(idx, size=max_per_class, replace=False)
#         indices.append(idx)
#     indices = np.concatenate(indices)
#     rng.shuffle(indices)
#     return X[indices], y[indices]

# X_data, y_data = balance_dataset(X_data, y_data, MAX_PER_CLASS)

# print(f"\nDataset balanceado: {X_data.shape[0]:,} muestras totales")
# for name, idx in GESTURE_MAP.items():
#     count = np.sum(y_data == idx)
#     print(f"  [{idx}] {name:<8s}  {count:,}")

# Train (80) / Test split (10) / Validation (10)

In [ ]:
# Primer split: 80% train, 20% temporal
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y_data,
)

# Segundo split: el 20% temporal → 10% test, 10% validacion
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp,
    test_size    = 0.50,
    random_state = 42,
    stratify     = y_temp,
)

print("Split 80 / 10 / 10 estratificado")
print(f"Train      : {len(X_train):,}  ({len(X_train)/len(X_data):.0%})")
print(f"Test       : {len(X_test):,}   ({len(X_test)/len(X_data):.0%})")
print(f"Validacion : {len(X_val):,}   ({len(X_val)/len(X_data):.0%})")

# Entrenamiento

Hiperparametros justificados:
- n_estimators=200- mayor ensemble da una menor varianza
- max_features='sqrt'- diversidad entre arboles, estandar para clasificacion
- min_samples_leaf=2- evita hojas de 1 muestra (ruido)
- class_weight='balanced'- compensa clases con diferente cantidad de imagenes
- n_jobs=-1- usa todos los nucleos disponibles

In [ ]:
#Para mi laptop
rf = RandomForestClassifier(
    n_estimators      = 400,
    max_features      = "sqrt",
    min_samples_split = 4,
    min_samples_leaf  = 1,
    class_weight      = None,  # datos ya balanceados
    random_state      = 42,
    n_jobs            = -1,
)

print("Entrenando Random Forest...")
t0 = time.time()
rf.fit(X_train, y_train)
print(f"Listo en {time.time() - t0:.1f}s")

# Evaluacion

In [ ]:
# Evaluacion 

y_pred = rf.predict(X_test)

f1_macro       = f1_score(y_test, y_pred, average="macro")
acc            = accuracy_score(y_test, y_pred)
f1_per_class   = f1_score(y_test, y_pred, average=None, labels=list(range(N_CLASSES)))
recall_per_class = recall_score(y_test, y_pred, average=None, labels=list(range(N_CLASSES)))
recall_alert   = recall_per_class[ALERT_CLASS]

print(f"F1-Score Macro : {f1_macro:.4f} (umbral >= {MVP_F1_MACRO})")
print(f"Accuracy       : {acc:.4f} (umbral >= {MVP_ACCURACY})")
print(f"Recall alerta  : {recall_alert:.4f} (umbral >= {MVP_ALERT_RECALL})")

print("\nF1 por gesto:")
for name, idx in GESTURE_MAP.items():
    bar   = "█" * int(f1_per_class[idx] * 30)
    alert = " <- ALERTA" if idx == ALERT_CLASS else ""
    print(f"  [{idx}] {name:<8s} {f1_per_class[idx]:.4f} {bar}{alert}")

print("\nReporte completo:")
print(classification_report(y_test, y_pred, target_names=LABEL_NAMES))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
ax.set_xlabel("Prediccion", fontsize=12)
ax.set_ylabel("Real", fontsize=12)
ax.set_title("Matriz de Confusion — GestoVoz", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

#### Validacion

In [ ]:
# ── Evaluacion en validacion ──────────────────────────────────────────────────
y_val_pred = rf.predict(X_val)
f1_val     = f1_score(y_val, y_val_pred, average="macro")
acc_val    = accuracy_score(y_val, y_val_pred)

print("VALIDACION")
print(f"  F1-Score Macro : {f1_val:.4f}")
print(f"  Accuracy       : {acc_val:.4f}")

diff_f1  = abs(f1_macro - f1_val)
diff_acc = abs(acc - acc_val)
print(f"\nDiferencia test vs val:")
print(f"  F1-Score : {diff_f1:.4f} {'estable' if diff_f1 < 0.03 else 'revisa overfitting'}")
print(f"  Accuracy : {diff_acc:.4f} {'estable' if diff_acc < 0.03 else 'revisa overfitting'}")

# Guardar modelo

In [ ]:
# import copy
# from coremltools.models.datatypes import Array

# # ── Guardar .pkl completo (modelo original intacto) ───────────────────────────
# joblib.dump(rf, MODEL_PATH)
# print(f"Modelo .pkl guardado : {MODEL_PATH}")

# ── Crear version lite para CoreML (50 arboles en vez de 300) ─────────────────
# No reentrenamos — solo tomamos los primeros 50 estimators del RF ya entrenado
# N_TREES_COREML = 50

# rf_lite = copy.deepcopy(rf)
# rf_lite.estimators_ = rf_lite.estimators_[:N_TREES_COREML]
# rf_lite.n_estimators = N_TREES_COREML

# print(f"\nRF lite: {N_TREES_COREML} arboles (de {rf.n_estimators} originales)")
# print("Verificando accuracy del modelo lite...")

# y_pred_lite = rf_lite.predict(X_test)
# acc_lite    = accuracy_score(y_test, y_pred_lite)
# f1_lite     = f1_score(y_test, y_pred_lite, average="macro")
# print(f"  Accuracy lite : {acc_lite:.4f}  (original: {acc:.4f})")
# print(f"  F1 Macro lite : {f1_lite:.4f}  (original: {f1_macro:.4f})")

# # ── Convertir RF lite a CoreML ────────────────────────────────────────────────
# print("\nIniciando conversion a Core ML (modelo lite)...")
# try:
#     modelo_coreml = ct.converters.sklearn.convert(
#         rf_lite,
#         input_features       = [("landmarks_42", Array(N_FEATURES))],
#         output_feature_names = "gesto_predicho",
#     )

#     modelo_coreml.author            = "Christofer Castaneda, Ivan Mijares, Alexander Orduna"
#     modelo_coreml.license           = "Computo Ubicuo — CETYS Universidad"
#     modelo_coreml.version           = "1.0-lite"
#     modelo_coreml.short_description = (
#         f"GestoVoz RF lite ({N_TREES_COREML} trees) — {N_CLASSES} gestos. "
#         f"Input: 42 floats (21 landmarks xy normalizados). "
#         f"F1 Macro: {round(f1_lite,4)} | Accuracy: {round(acc_lite,4)}"
#     )

#     MLMODEL_PATH = MODEL_PATH.replace(".pkl", ".mlmodel")
#     modelo_coreml.save(MLMODEL_PATH)
#     print(f"Modelo CoreML guardado: {MLMODEL_PATH}")

# except Exception as e:
#     print(f"Error: {e}")

# Modelo 3D

In [ ]:
N_COORDS_3D  = 3
N_FEATURES_3D = N_LANDMARKS * N_COORDS_3D + 5  # 63 coords + 5 angulos = 68

CACHE_X_3D = CACHE_DIR / "X_landmarks_3d.npy"
CACHE_Y_3D = CACHE_DIR / "y_labels_3d.npy"
MODEL_PATH_3D = "gestovoz_rf_3d.pkl"

print(f"Features 2D: {N_FEATURES}")
print(f"Features 3D: {N_FEATURES_3D}")

In [ ]:
# ── Normalizacion 3D ──────────────────────────────────────────────────────────
def normalize_landmarks_xyz(hand_landmarks) -> list[float]:
    raw_x = [lm.x for lm in hand_landmarks.landmark]
    raw_y = [lm.y for lm in hand_landmarks.landmark]
    raw_z = [lm.z for lm in hand_landmarks.landmark]

    # Centrar en muneca (landmark 0)
    wx, wy, wz = raw_x[0], raw_y[0], raw_z[0]
    cx = [x - wx for x in raw_x]
    cy = [y - wy for y in raw_y]
    cz = [z - wz for z in raw_z]

    # Normalizar escala con landmark 9 (base del dedo medio)
    scale = np.sqrt(cx[9]**2 + cy[9]**2 + cz[9]**2)
    if scale < 1e-6:
        scale = 1.0

    # 63 coordenadas normalizadas
    coords = []
    for x, y, z in zip(cx, cy, cz):
        coords.append(x / scale)
        coords.append(y / scale)
        coords.append(z / scale)

    # 5 angulos entre puntas de dedos (igual que 2D pero en 3D)
    fingertips = [4, 8, 12, 16, 20]
    angles = []
    for i in range(len(fingertips) - 1):
        a = fingertips[i]
        b = fingertips[i + 1]
        v1 = np.array([cx[a]/scale, cy[a]/scale, cz[a]/scale])
        v2 = np.array([cx[b]/scale, cy[b]/scale, cz[b]/scale])
        angles.append(angle_between(v1, v2))  # angle_between ya soporta 3D

    return coords + angles  # 68 floats

In [ ]:
# ── Extraccion y cache 3D ─────────────────────────────────────────────────────
if CACHE_X_3D.exists() and CACHE_Y_3D.exists():
    print("Cache 3D encontrado")
    X_data_3d = np.load(CACHE_X_3D)
    y_data_3d = np.load(CACHE_Y_3D)
    print(f"{X_data_3d.shape[0]:,} muestras | {X_data_3d.shape[1]} features")
else:
    print("Extrayendo landmarks 3D...\n")

    def process_image_3d(args):
        img_path, label = args
        mp_h = mp.solutions.hands
        with mp_h.Hands(
            static_image_mode=True,
            max_num_hands=1,
            min_detection_confidence=0.5
        ) as hands:
            img_bgr = cv2.imread(str(img_path))
            if img_bgr is None:
                return None
            result = hands.process(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
            if result.multi_hand_landmarks:
                feats = normalize_landmarks_xyz(result.multi_hand_landmarks[0])
                return (feats, label)
        return None

    def extract_dataset_3d(hagrid_root, gesture_map, max_per_class=None, workers=8):
        tasks = []
        for gesture_name, label in gesture_map.items():
            gesture_dir = hagrid_root / gesture_name
            images = sorted([p for p in gesture_dir.iterdir()
                             if p.suffix.lower() in EXTENSIONS])
            if max_per_class:
                images = images[:max_per_class]
            for img_path in images:
                tasks.append((img_path, label))

        X, y = [], []
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = {executor.submit(process_image_3d, t): t for t in tasks}
            for future in tqdm(as_completed(futures), total=len(tasks), desc="Extrayendo 3D"):
                result = future.result()
                if result is not None:
                    X.append(result[0])
                    y.append(result[1])
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

    X_data_3d, y_data_3d = extract_dataset_3d(HAGRID_ROOT, GESTURE_MAP, workers=8)
    np.save(CACHE_X_3D, X_data_3d)
    np.save(CACHE_Y_3D, y_data_3d)
    print(f"Cache 3D guardado en '{CACHE_DIR}'")

# Balanceo (mismo que 2D)
X_data_3d, y_data_3d = balance_dataset(X_data_3d, y_data_3d, MAX_PER_CLASS)
print(f"\nDataset 3D balanceado: {X_data_3d.shape[0]:,} muestras")

In [ ]:
# ── Entrenamiento modelo 3D ───────────────────────────────────────────────────
X_train_3d, X_temp_3d, y_train_3d, y_temp_3d = train_test_split(
    X_data_3d, y_data_3d, test_size=0.20, random_state=42, stratify=y_data_3d
)
X_test_3d, X_val_3d, y_test_3d, y_val_3d = train_test_split(
    X_temp_3d, y_temp_3d, test_size=0.50, random_state=42, stratify=y_temp_3d
)

rf_3d = RandomForestClassifier(
    n_estimators    = 400,   # ensemble grande → menor varianza
    max_features    = 'sqrt',# selección aleatoria → diversidad entre árboles
    min_samples_split = 4,   # evita divisiones con muy pocas muestras
    min_samples_leaf  = 1,   # hojas unitarias permitidas (datos balanceados)
    class_weight    = None,  # innecesario, el dataset ya fue balanceado
    random_state    = 42,    # reproducibilidad
    n_jobs          = -1,    # usa todos los núcleos disponibles

)

print("Entrenando Random Forest 3D...")
t0 = time.time()
rf_3d.fit(X_train_3d, y_train_3d)
print(f"Listo en {time.time() - t0:.1f}s")

joblib.dump(rf_3d, MODEL_PATH_3D)
print(f"Modelo 3D guardado: {MODEL_PATH_3D}")

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, recall_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Evaluacion 3D
y_pred_3d = rf_3d.predict(X_test_3d)

f1_macro_3d       = f1_score(y_test_3d, y_pred_3d, average="macro")
acc_3d            = accuracy_score(y_test_3d, y_pred_3d)
f1_per_class_3d   = f1_score(y_test_3d, y_pred_3d, average=None, labels=list(range(N_CLASSES)))
recall_per_class_3d = recall_score(y_test_3d, y_pred_3d, average=None, labels=list(range(N_CLASSES)))
recall_alert_3d   = recall_per_class_3d[ALERT_CLASS]

print(f"F1-Score Macro : {f1_macro_3d:.4f} (umbral >= {MVP_F1_MACRO})")
print(f"Accuracy       : {acc_3d:.4f} (umbral >= {MVP_ACCURACY})")
print(f"Recall alerta  : {recall_alert_3d:.4f} (umbral >= {MVP_ALERT_RECALL})")

print("\nF1 por gesto:")
for name, idx in GESTURE_MAP.items():
    bar   = "█" * int(f1_per_class_3d[idx] * 30)
    alert = " <- ALERTA" if idx == ALERT_CLASS else ""
    print(f"  [{idx}] {name:<8s} {f1_per_class_3d[idx]:.4f} {bar}{alert}")

print("\nReporte completo:")
print(classification_report(y_test_3d, y_pred_3d, target_names=LABEL_NAMES))

cm = confusion_matrix(y_test_3d, y_pred_3d)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
ax.set_xlabel("Prediccion", fontsize=12)
ax.set_ylabel("Real", fontsize=12)
ax.set_title("Matriz de Confusion 3D — GestoVoz", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("confusion_matrix_3d.png", dpi=150)
plt.show()

In [ ]:
# ── Comparacion 2D vs 3D ──────────────────────────────────────────────────────
y_pred_3d   = rf_3d.predict(X_test_3d)
f1_3d       = f1_score(y_test_3d, y_pred_3d, average="macro")
acc_3d      = accuracy_score(y_test_3d, y_pred_3d)

print("=" * 45)
print(f"{'Metrica':<20} {'2D':>10} {'3D':>10}")
print("=" * 45)
print(f"{'F1 Macro':<20} {f1_macro:>10.4f} {f1_3d:>10.4f}")
print(f"{'Accuracy':<20} {acc:>10.4f} {acc_3d:>10.4f}")
print("=" * 45)
delta_f1 = f1_3d - f1_macro
print(f"Mejora F1: {delta_f1:+.4f}")

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import joblib

# ── Cargar modelo 3D ──────────────────────────────────────────────────────────
MODEL_FILE = "gestovoz_rf_3d.pkl"
rf = joblib.load(MODEL_FILE)
print(f"Modelo cargado: {MODEL_FILE}")

# ── Config ────────────────────────────────────────────────────────────────────
GESTURE_MAP = {
    0: "dislike",
    1: "fist",
    2: "like",
    3: "ok",
    4: "one",
    5: "palm",
    6: "peace",
    7: "rock",
}
CONFIDENCE_THR = 0.80
ALERT_CLASS    = 5  # palm

mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# ── Normalizacion 3D ──────────────────────────────────────────────────────────
def normalize_3d(hand_landmarks):
    raw_x = [lm.x for lm in hand_landmarks.landmark]
    raw_y = [lm.y for lm in hand_landmarks.landmark]
    raw_z = [lm.z for lm in hand_landmarks.landmark]
    wx, wy, wz = raw_x[0], raw_y[0], raw_z[0]
    cx = [x - wx for x in raw_x]
    cy = [y - wy for y in raw_y]
    cz = [z - wz for z in raw_z]
    scale = np.sqrt(cx[9]**2 + cy[9]**2 + cz[9]**2) or 1.0
    coords = []
    for x, y, z in zip(cx, cy, cz):
        coords.extend([x / scale, y / scale, z / scale])
    fingertips = [4, 8, 12, 16, 20]
    angles = []
    for i in range(len(fingertips) - 1):
        a, b = fingertips[i], fingertips[i+1]
        v1 = np.array([cx[a]/scale, cy[a]/scale, cz[a]/scale])
        v2 = np.array([cx[b]/scale, cy[b]/scale, cz[b]/scale])
        dot  = np.dot(v1, v2)
        norm = np.linalg.norm(v1) * np.linalg.norm(v2)
        angles.append(np.arccos(np.clip(dot / norm, -1.0, 1.0)) if norm > 1e-6 else 0.0)
    return np.array(coords + angles, dtype=np.float32).reshape(1, -1)

# ── Loop de camara ────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    static_image_mode        = False,
    max_num_hands            = 1,
    min_detection_confidence = 0.60,
    min_tracking_confidence  = 0.50,
) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame  = cv2.flip(frame, 1)
        h, w   = frame.shape[:2]
        result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 80), (20, 20, 20), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

        cv2.putText(frame, "Modo: 3D", (w - 110, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 0), 2)

        if result.multi_hand_landmarks:
            hl = result.multi_hand_landmarks[0]

            mp_drawing.draw_landmarks(
                frame, hl, mp_hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 255, 180), thickness=2, circle_radius=3),
                mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2),
            )

            feats      = normalize_3d(hl)
            pred_class = rf.predict(feats)[0]
            proba      = rf.predict_proba(feats)[0]
            confidence = proba[pred_class]
            gesture    = GESTURE_MAP[pred_class]
            is_alert   = pred_class == ALERT_CLASS

            if is_alert:
                color = (0, 80, 255)
            elif confidence >= CONFIDENCE_THR:
                color = (0, 220, 100)
            else:
                color = (0, 165, 255)

            cv2.putText(frame, f"{gesture.upper()}  {confidence:.0%}", (16, 52),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.4, color, 3, cv2.LINE_AA)

            bar_w = int((w - 32) * confidence)
            cv2.rectangle(frame, (16, 62), (16 + bar_w, 72), color, -1)
            cv2.rectangle(frame, (16, 62), (w - 16, 72), (180, 180, 180), 1)

            if is_alert and confidence >= CONFIDENCE_THR:
                cv2.rectangle(frame, (w - 180, 90), (w - 10, 130), (0, 0, 200), -1)
                cv2.putText(frame, "ALERTA", (w - 170, 120),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)

            for rank, (gid, p) in enumerate(sorted(enumerate(proba), key=lambda x: -x[1])[:3]):
                cv2.putText(frame, f"{GESTURE_MAP[gid]:<10s} {p:.2f}",
                            (16, h - 70 + rank * 24),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                            (255, 255, 255) if gid == pred_class else (140, 140, 140), 1)
        else:
            cv2.putText(frame, "Sin mano detectada", (16, 52),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (100, 100, 100), 2)

        cv2.putText(frame, "Q: salir", (w - 100, h - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (160, 160, 160), 1)
        cv2.imshow("GestoVoz", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print("Camara cerrada")

Modelo cargado: gestovoz_rf_3d.pkl
Camara cerrada
